# 04 — Univariate Distributions
**Goal:** Choose and build the right chart when the question is *"how is one
variable distributed?"* — histogram, KDE, box, violin, strip, ECDF.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

np.random.seed(7)
print('numpy ', np.__version__)

## 1. The distribution family

Given one numeric variable, six core charts cover almost every need:

| Chart | Shows best | Watch out for |
|---|---|---|
| **Histogram** | Binned frequency | Bin size choice |
| **KDE** | Smooth shape | Can invent bumps; bandwidth sensitive |
| **Rug** | Raw data points alone | Useless past a few hundred points |
| **Box** | Median, IQR, outliers | Hides multimodality |
| **Violin** | Full shape per group | Confusing for non-statisticians |
| **Strip / swarm** | Every point | Slow above ~1000 points |
| **ECDF** | Exact percentiles | Less familiar to business readers |

In [ ]:
sample = np.concatenate([
    np.random.normal(0, 1, 400),
    np.random.normal(4, 0.5, 200),
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample, bins=30, color='steelblue', edgecolor='white')
ax.set_title('Histogram — bin size matters')
ax.set_xlabel('value'); ax.set_ylabel('count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, bins in zip(axes, [5, 80]):
    ax.hist(sample, bins=bins, color='steelblue', edgecolor='white')
    ax.set_title(f'bins={bins}')
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 2. Rules of thumb for histograms

1. **Sturges' rule** — `bins = ceil(log2(n) + 1)`. Conservative.
2. **Freedman–Diaconis** — `bins_width = 2 * IQR * n^(-1/3)`. Robust.
3. For >1000 points, prefer KDE to a histogram.
4. Always start the y-axis at zero (it is a count).

In [ ]:
n = len(sample)
iqr = np.subtract(*np.percentile(sample, [75, 25]))
fd_bin_w = 2 * iqr * n ** (-1/3)
fd_bins  = int(np.ceil((sample.max() - sample.min()) / fd_bin_w))
st_bins  = int(np.ceil(np.log2(n) + 1))
print(f'Sturges       : {st_bins} bins')
print(f'Freedman-Diac : {fd_bins} bins')

## 3. KDE — the smooth histogram

Kernel density estimation places a small bump (kernel) at every data point and
sums them. The bandwidth controls smoothness.

In [ ]:
x_grid = np.linspace(sample.min() - 1, sample.max() + 1, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, bw in zip(axes, [0.05, 0.3, 1.5]):
    kde = stats.gaussian_kde(sample, bw_method=bw)
    ax.plot(x_grid, kde(x_grid), color='steelblue', lw=2)
    ax.fill_between(x_grid, kde(x_grid), color='steelblue', alpha=0.3)
    ax.set_title(f'bandwidth = {bw}')
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Box plot anatomy

A box plot encodes **five numbers** plus outliers:

```
        whisker (max within 1.5*IQR)
        │
    ┌───┤
    │   │       ← IQR box (Q3 - Q1)
    │   │
    ├───┤       ← median (Q2)
    │   │
    │   │
    └───┘
        │
        whisker (min within 1.5*IQR)
  •     •       ← outliers beyond whiskers
```

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot(sample, vert=False, widths=0.5, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.5),
           medianprops=dict(color='crimson', linewidth=2))
ax.set_title('Box plot — 5-number summary + outliers')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 5. Violin vs strip vs swarm

When you want to see **every point** in a small dataset, strip and swarm plots
shine. Violin shows the *shape* without showing individual points.

In [ ]:
a = np.random.normal(0, 1, 80)
b = np.random.normal(1, 1.2, 80)
c = np.random.normal(2, 0.6, 80)
data = [a, b, c]
labels = ['A', 'B', 'C']

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
axes[0].violinplot(data, showmeans=True)
axes[0].set_xticks([1, 2, 3]); axes[0].set_xticklabels(labels)
axes[0].set_title('Violin — full shape')

for i, d in enumerate(data, start=1):
    axes[1].scatter([i]*len(d) + np.random.normal(0, 0.04, len(d)), d,
                    alpha=0.6, color='steelblue')
axes[1].set_xticks([1, 2, 3]); axes[1].set_xticklabels(labels)
axes[1].set_title('Strip — every point')

axes[2].boxplot(data, tick_labels=labels, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[2].set_title('Box — robust summary')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 6. The ECDF — the chart statisticians love

An empirical CDF answers the question *"what fraction of observations is at
most x?"* with zero binning, zero smoothing, zero assumptions.

In [ ]:
def ecdf(values):
    xs = np.sort(values)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys

fig, ax = plt.subplots(figsize=(8, 4))
for d, label in zip(data, ['A', 'B', 'C']):
    xs, ys = ecdf(d)
    ax.step(xs, ys, where='post', label=label, lw=2)
ax.set_xlabel('value'); ax.set_ylabel('F(x)')
ax.set_title('ECDF — exact cumulative distribution')
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Comparing groups — small multiples vs overlay

Two ways to compare distributions across categories:

1. **Overlay** — put them on the same axes, color-coded (works for 2–4 groups).
2. **Small multiples** — one panel per group (scales to any number).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['steelblue', 'crimson', 'seagreen']
for d, c, l in zip(data, colors, ['A', 'B', 'C']):
    axes[0].hist(d, bins=15, alpha=0.5, color=c, label=l, edgecolor='white')
axes[0].legend(); axes[0].set_title('Overlay')

for i, (d, c, l) in enumerate(zip(data, colors, ['A', 'B', 'C'])):
    axes[1].hist(d, bins=15, color=c, edgecolor='white')
    axes[1].set_title(l)
plt.tight_layout(); plt.show()

## 8. A real example — load data

Let's apply the family to a column from the unified marketing data.

In [ ]:
df = pd.read_csv('data/clean/unified_daily.csv')
print(df.columns.tolist())
print(df.shape)
df.head(3)

In [ ]:
num_col = 'spend' if 'spend' in df.columns else df.select_dtypes('number').columns[0]
print('Plotting:', num_col)
x = df[num_col].dropna().values

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes[0, 0].hist(x, bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Histogram')
kde = stats.gaussian_kde(x)
xg = np.linspace(x.min(), x.max(), 300)
axes[0, 1].plot(xg, kde(xg), color='steelblue')
axes[0, 1].set_title('KDE')
axes[1, 0].boxplot(x, vert=False, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[1, 0].set_title('Box')
xs, ys = ecdf(x)
axes[1, 1].step(xs, ys, where='post', color='steelblue')
axes[1, 1].set_title('ECDF')
for ax in axes.ravel():
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## Summary

| Chart | When to use |
|---|---|
| Histogram | First look at binned frequency |
| KDE | Smooth shape, multimodal detection |
| Box | Compact 5-number summary across groups |
| Violin | Show full shape across groups |
| Strip / swarm | Every point, small dataset |
| ECDF | Exact percentiles, no binning |

**Next:** `05_bivariate_relationships.ipynb` — two variables at a time.